# **COUNTY RAINFALL AND PLANTING ADVISORY**
## **PLP Data Analytics Assignment 2**

**Name:** Michelle Wambura Banda

**Cohort:** January 2026 PLP Data Analytics Cohort

This analysis cleans rainfall data, identifies rainfall patterns, and recommends planting windows for five counties.

In [23]:
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

## **Task 1: Cleaning Machakos Rainfal Data**
This section converts text values to integers and replaces invalid values using the average of valid rainfall readings.


In [24]:
months = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

In [25]:
valid_values = []
for value in machakos_raw:

    try:
        valid_values.append(
            int(value.strip())
        )

    except ValueError:
        pass

In [26]:
average_value = round(
    sum(valid_values)
    / len(valid_values)
)

print(average_value)

68


In [27]:
def clean_machakos(data):
    """
    Clean Machakos rainfall data and
    replace invalid values.
    """

    clean = []

    imputed_months = []

    for month, value in zip(
        months,
        data
    ):

        try:
            clean.append(
                int(value.strip())
            )

        except ValueError:

            clean.append(
                average_value
            )

            imputed_months.append(
                month
            )

    return clean, imputed_months

In [28]:
machakos_cleaned, imputed = clean_machakos(
    machakos_raw
)

In [29]:
print(
    "Imputed Months:"
)

for month in imputed:
    print(month)

Imputed Months:
April
September


## **TASK 2 - Build NumPy Array**

In [30]:
import numpy as np

all_rows = clean_rows + [
    machakos_cleaned
]

rainfall = np.array(
    all_rows
)
print(
    rainfall.shape
)

print(
    rainfall.dtype
)

(5, 12)
int64


# **TASK 3 - Totals and Monthly Averages**

In [31]:
annual_totals = rainfall.sum(
    axis=1
)
monthly_average = rainfall.mean(
    axis=0
)
print(
    annual_totals
)

print(
    monthly_average.round(2)
)


[1045  916 1371 1075  811]
[ 47.   47.6  91.  164.4 171.2  67.6  56.6  58.2  64.2  67.8 127.4  80.6]


## **TASK 4 - Wettest Month**

In [32]:
wettest_positions = np.argmax(
    rainfall,
    axis=1
)
wettest_months = [
    months[i]
    for i in wettest_positions
]
for county, month in zip(
    counties,
    wettest_months
):

    print(
        county,
        "->",
        month
    )

Nairobi -> April
Nakuru -> April
Kisumu -> April
Mombasa -> May
Machakos -> November


## **Task 5: Dry Month Analysis**

In [33]:
dry_mask = rainfall < 50
True
False
True
...
dry_months = dry_mask.sum(
    axis=1
)
for county, count in zip(
    counties,
    dry_months
):

    print(
        county,
        count
    )



Nairobi 4
Nakuru 3
Kisumu 0
Mombasa 2
Machakos 6


## **Task 6: Planting Windows**

In [34]:
pairs = (
    rainfall[:, :-1]
    +
    rainfall[:, 1:]
)
best_pair_index = np.argmax(
    pairs,
    axis=1
)

In [35]:
planting_windows = []

for position in best_pair_index:

    start_month = months[position]

    end_month = months[position + 1]

    planting_windows.append(
        f"{start_month}-{end_month}"
    )


In [36]:
for county, window in zip(
    counties,
    planting_windows
):

    print(
        county,
        "->",
        window
    )

Nairobi -> April-May
Nakuru -> April-May
Kisumu -> April-May
Mombasa -> April-May
Machakos -> November-December


## **Task 7: Export Summary CSV**

In [37]:
import csv
with open(
    "rainfall_summary.csv",
    "w",
    newline=""
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "county",
            "annual_total_mm",
            "wettest_month",
            "dry_months",
            "planting_window"
        ]
    )

    writer.writeheader()

    for county, total, wettest, dry, window in zip(
        counties,
        annual_totals,
        wettest_months,
        dry_months,
        planting_windows
    ):

        writer.writerow({
            "county": county,
            "annual_total_mm": total,
            "wettest_month": wettest,
            "dry_months": dry,
            "planting_window": window
        })


In [38]:
with open(
    "rainfall_summary.csv"
) as file:

    print(
        file.read()
    )

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April-May
Nakuru,916,April,3,April-May
Kisumu,1371,April,0,April-May
Mombasa,1075,May,2,April-May
Machakos,811,November,6,November-December



## **Planting Advisory**

The analysis shows that most counties receive their highest rainfall during April and May, making this period suitable for planting. Kisumu recorded the highest annual rainfall totals, indicating strong potential for rain-fed agriculture. Several counties experience dry months with rainfall below 50 mm, suggesting a need for water conservation and drought preparedness strategies. The recommended planting windows are based on the two consecutive months with the highest combined rainfall. Machakos contained missing rainfall observations that were replaced using the average of valid values, so results for that county should be interpreted with caution.

## **Data Quality Notes**
Machakos rainfall data contained invalid entries including "N/A" and blank values. These values were replaced with the rounded average of valid rainfall observations. While this preserves the dataset for analysis, the imputed values may not perfectly represent actual rainfall during the missing months.